# Session 8. Deploy: FastAPI, Docker Compose, Telegram

**The graph leaves the notebook and gets an address.**

- every code cell here runs offline; Postgres and Telegram are wired live in class
- one env var switches the notebook lane to the compose lane
- milestone session: by the end of practice your assistant answers in Telegram

In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()  # reads .env once; nothing below opens a file


def chat_model(size: str = "cheap", **kwargs):
    """A model object for the configured provider. A dozen lines, copy them once."""
    name = os.environ[f"MODEL_{size.upper()}"]  # ids live in .env, never in code
    secret = os.environ["LLM_API_KEY"]
    if os.getenv("LLM_REASONING_EFFORT"):  # gpt-5.x: tools need reasoning "none"
        kwargs.setdefault("reasoning_effort", os.environ["LLM_REASONING_EFFORT"])
    # Gemini over OpenAI-compat drops the reasoning signature: turn two 400s
    if os.getenv("LLM_PROVIDER", "openai_compat") == "google_genai":
        return init_chat_model(f"google_genai:{name}", api_key=secret, **kwargs)
    return init_chat_model(
        f"openai:{name}", api_key=secret, base_url=os.environ["LLM_BASE_URL"], **kwargs
    )


print("provider:", os.getenv("LLM_PROVIDER", "openai_compat"),
      "| strong:", os.environ["MODEL_STRONG"])

## The production shape

**pmtool-ai runs in production on pieces you already know.**

- FastAPI services around compiled graphs, about twenty containers in one compose file
- LiteLLM as the single OpenAI-compatible door to every model
- self-hosted Langfuse reading every trace: session 2's stack, instrumented since session 7, unchanged
- today rebuilds the teachable core: api, db, langfuse

**LiteLLM is for one person switching channels, not only for platforms.**

- the course hands every student a faculty-paid key
- your code calls one OpenAI-compatible address; swapping providers is a config edit
- the provider-switching appendix shows the config; no LiteLLM code today

**Why this course builds its own deploy stack.**

- the `langgraph` library is MIT: yours for anything, forever
- `langgraph-api`, the server behind `langgraph up`, is Elastic License 2.0: production wants a commercial key
- Aegra reimplements the same API under Apache-2.0, if you ever need that shape
- FastAPI plus Postgres plus compose covers everything this course ships

## Parcel tools

**A parcel outlives a process. The dialog about it must too.**

- `track_parcel` returns a status and a carrier code; `carrier_contact` takes that code
- dependent calls again: the second argument comes from the first result
- in-memory dicts, no network: today is about the wrapping, not the tools

In [ ]:
from langchain_core.tools import tool

# a fake carrier network: no HTTP, same answers every run
PARCELS = {
    "RR-1001": ("in transit, left the sorting hub", "CDE"),
    "RR-2002": ("held at customs, papers requested", "GLX"),
    "RR-3003": ("delivered, signed by the recipient", "CDE"),
}
CARRIERS = {
    "CDE": "City Delivery Express: support 09:00-18:00 weekdays, chat on the site",
    "GLX": "GlobalLux: support around the clock, answers within a day",
}


@tool  # the docstring is the description the model reads
def track_parcel(tracking_id: str) -> str:
    """Return the status and carrier code for a tracking id like RR-1001."""
    key = tracking_id.strip().upper()
    if key not in PARCELS:
        # a miss the model can act on, not an exception
        return f"Unknown id {tracking_id!r}. Known demo ids: {', '.join(sorted(PARCELS))}."
    status, carrier = PARCELS[key]
    return f"{key}: {status}. Carrier code: {carrier}."


@tool
def carrier_contact(carrier: str) -> str:
    """Return support contact hours for a carrier code from track_parcel."""
    code = carrier.strip().upper()
    if code not in CARRIERS:
        return f"Unknown carrier {carrier!r}. Call track_parcel first for a valid code."
    return CARRIERS[code]


TOOLS = [track_parcel, carrier_contact]
print(track_parcel.invoke({"tracking_id": "rr-1001"}))
print(track_parcel.invoke({"tracking_id": "XX-9"}))  # a sentence, not a crash

**All of session 3 in one call, smoke-tested before packaging.**

- `create_agent`, a system prompt, two tools, an explicit `recursion_limit`
- no checkpointer yet: it arrives with the service, where it belongs

In [ ]:
from langchain.agents import create_agent

PROMPT = (
    "You are a parcel support assistant. Use track_parcel for status, "
    "then carrier_contact for how to reach the carrier."
)

agent = create_agent(chat_model("cheap"), TOOLS, system_prompt=PROMPT)

smoke = agent.invoke(
    {"messages": [{"role": "user", "content": "Where is parcel RR-1001?"}]},
    config={"recursion_limit": 8},  # session-2 discipline, unchanged
)
print(smoke["messages"][-1].content)
print(agent.get_graph().draw_mermaid())  # the shape everything below wraps

## A checkpointer that survives restarts

**The graph is finished. Everything below this line is packaging.**

- the dialog state must outlive the process: Postgres, via `AsyncPostgresSaver`
- the notebook lane keeps `InMemorySaver`, so every cell still runs offline
- one env var, `DATABASE_URL`, picks the lane inside the app's lifespan

In [ ]:
from contextlib import asynccontextmanager

from fastapi import FastAPI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver

MEMORY = InMemorySaver()  # notebook lane: lives exactly as long as this process


def build_graph(checkpointer):
    return create_agent(chat_model("cheap"), TOOLS, system_prompt=PROMPT,
                        checkpointer=checkpointer)


@asynccontextmanager
async def lifespan(app: FastAPI):
    url = os.environ.get("DATABASE_URL")  # set by compose, absent in the notebook
    if url:
        async with AsyncPostgresSaver.from_conn_string(url) as saver:
            await saver.setup()  # creates the checkpoint tables on first start
            app.state.graph = build_graph(saver)
            yield  # the connection stays open for the app's whole life
    else:
        app.state.graph = build_graph(MEMORY)
        yield


print("lane:", "postgres" if os.environ.get("DATABASE_URL") else "in-memory")

**`setup()` runs once, then the tables exist. Versions bite here.**

- `await checkpointer.setup()` creates the checkpoint schema on first start
- the install is `psycopg[binary,pool]` plus `langgraph-checkpoint-postgres`; without the binary wheel the import dies hunting libpq
- interface package langgraph-checkpoint is 4.x; the postgres saver versions independently
- offline the Postgres branch is imported and compiled, never entered: by design

## The streaming endpoint

**One route, POST /chat/{thread_id}. The reply is a stream.**

- SSE: `data:` lines over one kept-open response, the shape every chat product uses
- `stream_mode="messages"` yields each message with the node that produced it
- the endpoint forwards only what the model node says

In [ ]:
from fastapi.responses import StreamingResponse
from pydantic import BaseModel

app = FastAPI(title="parcel-agent", lifespan=lifespan)
CALLBACKS: list = []  # observability plugs in here, later today


class ChatIn(BaseModel):
    text: str


@app.post("/chat/{thread_id}")
async def chat(thread_id: str, body: ChatIn) -> StreamingResponse:
    config = {
        "configurable": {"thread_id": thread_id},  # the URL names the dialog
        "recursion_limit": 8,
        "callbacks": CALLBACKS,
    }

    async def sse():
        stream = app.state.graph.astream(
            {"messages": [{"role": "user", "content": body.text}]},
            config=config,
            stream_mode="messages",  # (message, metadata) pairs as they happen
        )
        async for message, metadata in stream:
            if metadata.get("langgraph_node") == "model" and message.content:
                yield f"data: {message.content}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(sse(), media_type="text/event-stream")


print("routes:", [route.path for route in app.routes if "chat" in route.path])

**The URL names the thread. The server itself remembers nothing.**

- the checkpointer owns every dialog, keyed by `thread_id`; kill the process, checkpoints stay
- drop the node filter and tool payloads stream too: a handy debug trick
- the docs cap `thread_id` at 255 chars, so keep ids short; numeric chat ids never come close
- `data: [DONE]` is our end sentinel, the same convention big providers use

## Two turns through the service

**A real HTTP round trip, in-process, no sockets.**

- entering `TestClient` runs the lifespan: the lane gets chosen right here
- offline that means `InMemorySaver`; in the compose lane the same cells hit Postgres

In [ ]:
from fastapi.testclient import TestClient

with TestClient(app) as client:  # entering runs the lifespan: the lane is chosen
    with client.stream(
        "POST", "/chat/tg-314159", json={"text": "Where is parcel RR-1001?"}
    ) as reply:
        print("status:", reply.status_code, "|", reply.headers["content-type"])
        for line in reply.iter_lines():
            if line:
                print(line)

**One data line, then the sentinel.**

- the scripted model returns whole messages; a live model fills this stream token by token
- both tool calls ran and stayed server-side: the filter passed only the model's words
- a checkpoint was written after every step while this streamed, exactly as session 4 taught

**RAM forgets. That is the entire reason today exists.**

- `InMemorySaver` dies with its process, and every dialog dies with it
- with Postgres the same cells survive `docker compose restart api` mid-dialog
- that restart drill is practice step 5: run it while your bot is mid-conversation

In [ ]:
with TestClient(app) as client:  # a second server lifetime, same process
    with client.stream(
        "POST", "/chat/tg-314159", json={"text": "Has it moved? How do I reach them?"}
    ) as reply:
        for line in reply.iter_lines():
            if line:
                print(line)

    # read history while the server lives: the Postgres saver needs its loop
    state = app.state.graph.get_state({"configurable": {"thread_id": "tg-314159"}})
    print(len(state.values["messages"]), "messages on the thread")  # one dialog
    print("turn 1 opened with:", state.values["messages"][0].content)

**Twelve messages, and no request carried them.**

- turn 2 sent nothing but new text and the same thread id
- the saver outlived both server lifetimes because it lives outside them
- swap the lane and "outside" means a database volume instead of process RAM

## The compose stack

**Three services, one file, one `.env` read from both sides.**

- the api waits for a healthy Postgres, not a started one
- compose passes `DATABASE_URL`, so the lifespan takes the Postgres branch
- the deploy-template carries the full file; this is its readable core

The whole service side of today, one file:

```yaml
include:
  - path:                         # session 2's langfuse stack, plus its key-bootstrap override
      - docker-compose.langfuse.yml
      - docker-compose.override.yml

services:
  api:
    build: .
    env_file: .env                # compose and the code read the same file
    environment:
      DATABASE_URL: postgresql://postgres:${POSTGRES_PASSWORD}@db:5432/postgres
    ports:
      - "8000:8000"
    depends_on:
      db:
        condition: service_healthy   # ready, not merely started

  db:                             # named db: langfuse's own postgres keeps its name and data
    image: postgres:17
    environment:
      POSTGRES_PASSWORD: ${POSTGRES_PASSWORD}
    volumes:
      - parcel-pgdata:/var/lib/postgresql/data
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U postgres"]
      interval: 3s
      retries: 10

volumes:
  parcel-pgdata:
```

- `${POSTGRES_PASSWORD}` comes from the same `.env` your code reads: one file, both sides
- the include is session 2's stack: rename its downloaded file to `docker-compose.langfuse.yml`, keep its override beside it
- our checkpoint database is `db`, never `postgres`: langfuse's stack already owns that service name

## The Telegram bot

**Long polling pulls updates outward. No public address needed.**

- the bot keeps asking Telegram "anything new?", so it runs from any laptop
- webhooks are the inverse: Telegram calls you, and that needs a public URL
- ten lines of aiogram; the graph already does everything else

In [ ]:
from aiogram import Bot, Dispatcher
from aiogram.types import Message

bot = Bot(token=os.environ.get("TELEGRAM_BOT_TOKEN") or "1:offline")  # no network yet
dp = Dispatcher()


@dp.message()
async def on_message(message: Message) -> None:
    config = {"configurable": {"thread_id": str(message.chat.id)}, "recursion_limit": 8}
    result = await app.state.graph.ainvoke(
        {"messages": [{"role": "user", "content": message.text}]}, config=config
    )
    await message.answer(result["messages"][-1].content)  # the graph's final reply


print("handlers registered:", len(dp.message.handlers))
# dp.run_polling(bot) is all bot.py adds; polling starts there, never here

**`thread_id = str(chat_id)`: one line makes Telegram the session store.**

- every chat is a thread; a group chat shares one thread, feature and bug
- here the handler calls the graph directly; the template routes the bot through the api service, so both doors share one checkpointer
- in compose the bot is its own service on the api image:

```yaml
  bot:
    build: .
    command: python bot.py
    env_file: .env
    depends_on: [api]
```

**A bot is a token. Guard it accordingly.**

- message @BotFather, send `/newbot`, answer two questions; any Telegram account works
- `TELEGRAM_BOT_TOKEN` goes into `.env` and nowhere else, like every key this course touches
- a leaked token means someone else is your bot, to every user it has

In [ ]:
from langfuse import get_client
from langfuse.langchain import CallbackHandler

lf = get_client()  # reads LANGFUSE_HOST and both keys from the environment
print("server:", os.getenv("LANGFUSE_HOST"), "| up:", lf.auth_check())

CALLBACKS.append(CallbackHandler())  # every /chat request now writes one trace

with TestClient(app) as client:
    client.post("/chat/traced-thread", json={"text": "Where is parcel RR-2002?"})

lf.flush()  # in the container, lifespan shutdown calls this instead

**Inside a container, the trace is the only debugger left.**

- the same CallbackHandler as session 7, one trace per chat turn
- in the template it is pre-wired, and lifespan shutdown calls `flush()`
- no more print-and-rerun: real users are generating your logs now

## The deploy template

**The course template is this notebook's code, nothing else.**

- the reference repository at tag `session-08`: these cells as files, plus a Dockerfile
- every line of it was read in a lecture; you can defend all of it
- the infrastructure is shared; the agent behind it stays yours
- production extras are out of scope: see the last card

## Practice

**Your own agent, live in Telegram, from your own repo.**

1. fork the course template and drop in your own graph: begun in session 3, carrying everything since
2. `docker compose up`; watch `setup()` create the checkpoint tables on first start
3. token from @BotFather into `.env`; commit `.env.example`, never `.env`
4. talk to your bot in Telegram, then find that turn's trace

**Prove the state survives. Bring the evidence.**

5. restart drill: `docker compose restart api` mid-dialog; the bot must still remember
6. export the trace of one Telegram turn from your Langfuse
7. stretch: move the stack to a rented VPS; ruble-billed hosts work fine

**Required artifact: `runs/session-08.md`, committed.**

- bot username
- transcripts from before and after the restart drill
- the exported trace of one Telegram turn

**Mid-course milestone: the teaching product is deployed.**

- the session closes with the room talking to each other's bots
- by defense the project must run under compose, on any hosting
- the interface is your choice: Telegram bot, web chat over FastAPI, any live client

## Not taught today

**Left out on purpose; the template does not carry them either.**

- LiteLLM configuration: the provider-switching appendix covers it
- Aegra, when you want the managed-server API without the license
- JWT auth, rate limits, migrations, metrics: production concerns; read them in
  https://github.com/wassim249/fastapi-langgraph-agent-production-ready-template

## Next time

**Multi-agent systems: parallel branches, subgraphs, and when one loop beats them.**